In [1]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [1]:
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch

model_name = "Qwen/Qwen3-4B"
base_model = AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.float16,
    device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████| 398/398 [00:20<00:00, 19.07it/s]


In [2]:
from peft import PeftModel

# 2. Load the PEFT model with adapter
peft_model = PeftModel.from_pretrained(
    base_model, "/home/tianjp/llmLearn/stf/Qwen3-4B/Qlora/best", dtype=torch.float16
)


In [3]:
merged_model = peft_model.merge_and_unload()

In [4]:
prompt = "抽取出文本中的关键词：\n标题：人工神经网络在猕猴桃种类识别上的应用\n文本：在猕猴桃介电特性研究的基础上,将人工神经网络技术应用于猕猴桃的种类识别.该种类识别属于模式识别,其关键在于提取样品的特征参数,在获得特征参数的基础上,选取合适的网络通过训练来进行识别.猕猴桃种类识别的研究为自动化识别果品的种类、品种和新鲜等级等提供了一种新方法,为进一步研究果品介电特性与其内在品质的关系提供了一定的理论与实践基础."

messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
print(text)

<|im_start|>user
抽取出文本中的关键词：
标题：人工神经网络在猕猴桃种类识别上的应用
文本：在猕猴桃介电特性研究的基础上,将人工神经网络技术应用于猕猴桃的种类识别.该种类识别属于模式识别,其关键在于提取样品的特征参数,在获得特征参数的基础上,选取合适的网络通过训练来进行识别.猕猴桃种类识别的研究为自动化识别果品的种类、品种和新鲜等级等提供了一种新方法,为进一步研究果品介电特性与其内在品质的关系提供了一定的理论与实践基础.<|im_end|>
<|im_start|>assistant
<think>

</think>




In [5]:
model_inputs = tokenizer([text], return_tensors="pt").to(merged_model.device)

# conduct text completion
generated_ids = merged_model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: 
content: 猕猴桃;介电特性;神经网络;种类识别


In [6]:
merged_model.device

device(type='cuda', index=0)

In [7]:
merged_model.save_pretrained("/home/tianjp/llmLearn/stf/Qwen3-4B/QLoRa/merged")
tokenizer.save_pretrained("/home/tianjp/llmLearn/stf/Qwen3-4B/QLoRa/merged")

Writing model shards: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [04:45<00:00, 285.11s/it]


('/home/tianjp/llmLearn/stf/Qwen3-4B/QLoRa/merged/tokenizer_config.json',
 '/home/tianjp/llmLearn/stf/Qwen3-4B/QLoRa/merged/chat_template.jinja',
 '/home/tianjp/llmLearn/stf/Qwen3-4B/QLoRa/merged/tokenizer.json')